In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ========= 参数区（按需修改） =========
# 支持多个模型：键名用于当前目录下输出子文件夹命名
# model_roots = {
#     "AlphaFold3_v3.0.1_msa_notemp": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/PepSet_dimer_all"),
#     "Protenix_v1.0.0_msa_notemp": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_msa_notemplate-pep_nomsa_notemplate"),
#     "Protenix_v1.0.0_msa_pep-notemp": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_msa_template-pep_nomsa_notemplate"),
#     "Protenix_v1.0.0_msa_temp": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_msa_template-pep_nomsa_template"),
#     "Protenix_v1.0.0_nomsa_temp": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_nomsa_template-pep_nomsa_notemplate"),
#     "Protenix_v0.5.0_msa_notemp": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred-v05-pro_msa_notemplate-pep_nomsa_notemplate")
# }
model_roots = {
    "AlphaFold3_v3.0.1-pro_msa_template": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/pro_msa_template-pep_nomsa_notemplate"),
    "AlphaFold3_v3.0.1-pro_msa_notemplate": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/pro_msa_notemplate-pep_nomsa_notemplate/output"),
    "AlphaFold3_v3.0.1-pro_nomsa_template": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/pro_nomsa_template-pep_nomsa_notemplate/output"),
    "AlphaFold3_v3.0.1-pro_nomsa_notemplate": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/pro_nomsa_notemplate-pep_nomsa_notemplate/output"),

    "Protenix_v1.0.0-pro_msa_template": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_msa_template-pep_nomsa_notemplate"),
    "Protenix_v1.0.0-pro_msa_notemplate": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_msa_notemplate-pep_nomsa_notemplate"),
    "Protenix_v1.0.0-pro_nomsa_template": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_nomsa_template-pep_nomsa_notemplate"),
    "Protenix_v1.0.0-pro_nomsa_notemplate": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_nomsa_notemplate-pep_nomsa_notemplate"),


    # "Protenix_v0.5.0-pro_msa_notemplate": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred-v05-pro_msa_notemplate-pep_nomsa_notemplate"),
    # "Protenix_v1.0.0-pro_msa_template-pep_nomsa_template": Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/pred_v1-pro_msa_template-pep_nomsa_template")


   
}
# 仅处理该列表中的父PDB
pdb_list_path = Path("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/dataset/PepSet_dimer/PDB.list")

# 固定规则：每个(method, parent_pdb)必须15个样本（3 seeds × 5 id），如果后续增加seeds，则随之调整
required_samples_per_group = 15

# ===== 新增：绘图可配置参数 =====
axis_title_fontsize = 13
plot_title_fontsize = 15
tick_label_fontsize = 10
method_label_wrap_width = 14
x_tick_rotation = 0

# ===== 新增：测试模式（只绘制一个父PDB） =====
test_only_single_pdb = False
test_parent_pdb = "1a0n"

# 当前notebook所在目录作为工作目录
workdir = Path.cwd()
run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
output_date = run_tag[:8]

# 输出目录
logs_dir = workdir / "logs"
figures_dir = workdir / "figures"
exports_dir = workdir / "exports"
exports_date_dir = exports_dir / output_date
csv_summary_root_dir = workdir / "csv_summary"
csv_summary_dir = csv_summary_root_dir / output_date
for p in (logs_dir, figures_dir, exports_dir, exports_date_dir, csv_summary_root_dir, csv_summary_dir):
    p.mkdir(parents=True, exist_ok=True)


def load_pdb_list(path: Path) -> list[str]:
    if not path.exists():
        raise FileNotFoundError(f"pdb.list not found: {path}")
    pdbs = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                pdbs.append(line)
    return sorted(set(pdbs))


pdb_ids = load_pdb_list(pdb_list_path)
print(f"Loaded PDB count: {len(pdb_ids)}")
print("Methods:", ", ".join(model_roots.keys()))
print(f"Workdir: {workdir}")
print(f"csv_summary_dir: {csv_summary_dir}")
print(f"exports_date_dir: {exports_date_dir}")
print(f"axis_title_fontsize={axis_title_fontsize}, plot_title_fontsize={plot_title_fontsize}, tick_label_fontsize={tick_label_fontsize}")
print(f"x_tick_rotation={x_tick_rotation}, method_label_wrap_width={method_label_wrap_width}")
print(f"test_only_single_pdb={test_only_single_pdb}, test_parent_pdb={test_parent_pdb}")

Loaded PDB count: 170
Methods: AlphaFold3_v3.0.1-pro_msa_template, AlphaFold3_v3.0.1-pro_msa_notemplate, AlphaFold3_v3.0.1-pro_nomsa_template, AlphaFold3_v3.0.1-pro_nomsa_notemplate, Protenix_v1.0.0-pro_msa_template, Protenix_v1.0.0-pro_msa_notemplate, Protenix_v1.0.0-pro_nomsa_template, Protenix_v1.0.0-pro_nomsa_notemplate, Protenix_v0.5.0-pro_msa_notemplate
Workdir: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary
csv_summary_dir: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/csv_summary/20260327
exports_date_dir: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/exports/20260327
axis_title_fontsize=13, plot_title_fontsize=15, tick_label_fontsize=10
x_tick_rotation=0, method_label_wrap_width=14
test_only_single_pdb=False, test_parent_pdb=1a0n


In [3]:
# ========= Step 1: 复制并重命名 metrics_summary.csv =========
copy_records = []
missing_records = []

for method, root in model_roots.items():
    root = Path(root)
    method_dir = csv_summary_dir / method
    method_dir.mkdir(parents=True, exist_ok=True)

    if not root.exists():
        for pdb in pdb_ids:
            missing_records.append({
                "method": method,
                "parent_pdb": pdb,
                "reason": "method_root_not_found",
                "source": str(root / pdb / "metrics_summary.csv"),
            })
        continue

    for pdb in pdb_ids:
        src = root / pdb / "metrics_summary.csv"
        dst = method_dir / f"{pdb}.metrics_summary.csv"

        if src.exists():
            shutil.copy2(src, dst)
            copy_records.append({
                "method": method,
                "parent_pdb": pdb,
                "source": str(src),
                "dest": str(dst),
            })
        else:
            missing_records.append({
                "method": method,
                "parent_pdb": pdb,
                "reason": "metrics_summary_missing",
                "source": str(src),
            })

copy_df = pd.DataFrame(copy_records)
missing_df = pd.DataFrame(missing_records)

copied_by_method = (
    copy_df.groupby("method")["parent_pdb"].nunique().rename("copied")
    if not copy_df.empty
    else pd.Series(dtype="int64", name="copied")
)

missing_by_method = (
    missing_df.groupby("method")["parent_pdb"].nunique().rename("missing")
    if not missing_df.empty
    else pd.Series(dtype="int64", name="missing")
)

copy_summary = (
    copied_by_method.to_frame()
    .join(missing_by_method, how="outer")
    .fillna(0)
    .astype(int)
    .reset_index()
)

if copy_summary.empty:
    copy_summary = pd.DataFrame({
        "method": list(model_roots.keys()),
        "copied": [0] * len(model_roots),
        "missing": [0] * len(model_roots),
    })

copy_summary["requested"] = len(pdb_ids)

copy_detail_path = exports_date_dir / f"copy_detail_{run_tag}.csv"
missing_log_path = logs_dir / f"copy_missing_{run_tag}.csv"
copy_summary_path = exports_date_dir / f"copy_summary_{run_tag}.csv"

copy_df.to_csv(copy_detail_path, index=False)
missing_df.to_csv(missing_log_path, index=False)
copy_summary.to_csv(copy_summary_path, index=False)

print("Copy summary:")
print(copy_summary)
print(f"Saved: {copy_detail_path}")
print(f"Saved: {missing_log_path}")
print(f"Saved: {copy_summary_path}")

Copy summary:
                                   method  copied  missing  requested
0    AlphaFold3_v3.0.1-pro_msa_notemplate     170        0        170
1      AlphaFold3_v3.0.1-pro_msa_template     170        0        170
2  AlphaFold3_v3.0.1-pro_nomsa_notemplate     170        0        170
3    AlphaFold3_v3.0.1-pro_nomsa_template     170        0        170
4      Protenix_v0.5.0-pro_msa_notemplate     170        0        170
5      Protenix_v1.0.0-pro_msa_notemplate     170        0        170
6        Protenix_v1.0.0-pro_msa_template     170        0        170
7    Protenix_v1.0.0-pro_nomsa_notemplate     170        0        170
8      Protenix_v1.0.0-pro_nomsa_template     170        0        170
Saved: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/exports/20260327/copy_detail_20260327_152811.csv
Saved: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/logs/copy_missing_20260327_152811.csv
Saved: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/e

In [4]:
# ========= Step 2: 读取复制结果并做15样本QC =========
required_cols = {"complex", "seed", "id", "scRMSD"}
read_records = []
bad_schema_records = []
duplicate_records = []

for method in model_roots.keys():
    method_dir = csv_summary_dir / method
    if not method_dir.exists():
        continue

    for csv_file in sorted(method_dir.glob("*.metrics_summary.csv")):
        parent_pdb = csv_file.name.replace(".metrics_summary.csv", "")
        try:
            df = pd.read_csv(csv_file)
        except Exception as e:
            bad_schema_records.append({
                "method": method,
                "parent_pdb": parent_pdb,
                "source_file": str(csv_file),
                "reason": f"read_error: {e}",
            })
            continue

        missing_cols = sorted(required_cols - set(df.columns))
        if missing_cols:
            bad_schema_records.append({
                "method": method,
                "parent_pdb": parent_pdb,
                "source_file": str(csv_file),
                "reason": f"missing_columns: {missing_cols}",
            })
            continue

        key_cols = ["complex", "seed", "id"]
        dup_count = int(df.duplicated(subset=key_cols).sum())
        if dup_count > 0:
            duplicate_records.append({
                "method": method,
                "parent_pdb": parent_pdb,
                "source_file": str(csv_file),
                "duplicate_rows": dup_count,
            })
            df = df.drop_duplicates(subset=key_cols, keep="first")

        df = df.copy()
        df["method"] = method
        df["parent_pdb"] = parent_pdb
        df["source_file"] = str(csv_file)
        read_records.append(df)

if not read_records:
    raise RuntimeError("No valid metrics_summary.csv loaded. Please check model_roots and pdb.list.")

all_df = pd.concat(read_records, ignore_index=True)
all_df["scRMSD"] = pd.to_numeric(all_df["scRMSD"], errors="coerce")
all_df = all_df.dropna(subset=["scRMSD"])

group_counts = (
    all_df.groupby(["method", "parent_pdb"], as_index=False)
    .size()
    .rename(columns={"size": "sample_count"})
)
group_counts["is_complete_15"] = group_counts["sample_count"] == required_samples_per_group

valid_groups = group_counts[group_counts["is_complete_15"]][["method", "parent_pdb"]]
incomplete_groups = group_counts[~group_counts["is_complete_15"]].copy()

valid_df = all_df.merge(valid_groups, on=["method", "parent_pdb"], how="inner")

bad_schema_df = pd.DataFrame(bad_schema_records)
duplicate_df = pd.DataFrame(duplicate_records)

qc_summary_path = exports_date_dir / f"group_qc_summary_{run_tag}.csv"
incomplete_log_path = logs_dir / f"incomplete_pdb_{run_tag}.csv"
bad_schema_log_path = logs_dir / f"bad_schema_{run_tag}.csv"
duplicate_log_path = logs_dir / f"duplicates_{run_tag}.csv"
all_df_path = exports_date_dir / f"all_loaded_samples_{run_tag}.csv"
valid_df_path = exports_date_dir / f"scRMSD_valid_samples_{run_tag}.csv"

group_counts.to_csv(qc_summary_path, index=False)
incomplete_groups.to_csv(incomplete_log_path, index=False)
bad_schema_df.to_csv(bad_schema_log_path, index=False)
duplicate_df.to_csv(duplicate_log_path, index=False)
all_df.to_csv(all_df_path, index=False)
valid_df.to_csv(valid_df_path, index=False)

print(f"Loaded rows: {len(all_df)}")
print(f"Valid rows (complete groups only): {len(valid_df)}")
print(f"Complete groups: {len(valid_groups)}")
print(f"Incomplete groups: {len(incomplete_groups)}")
print(f"Saved: {qc_summary_path}")
print(f"Saved: {incomplete_log_path}")
print(f"Saved: {valid_df_path}")

Loaded rows: 22950
Valid rows (complete groups only): 22950
Complete groups: 1530
Incomplete groups: 0
Saved: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/exports/20260327/group_qc_summary_20260327_152811.csv
Saved: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/logs/incomplete_pdb_20260327_152811.csv
Saved: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/exports/20260327/scRMSD_valid_samples_20260327_152811.csv


In [5]:
# ========= Step 3: 绘制scRMSD箱线图（按PDB分图 + Top1/Oracle全局总图） =========
import pandas as pd
sns.set_theme(style="whitegrid", context="talk")

# Step3 模式: "full" = 绘图+计算, "df_only" = 仅计算dataframe（不绘图，供 Step4 使用）
step3_mode = "df_only"

if valid_df.empty:
    raise RuntimeError("No complete (15-sample) groups available for plotting.")


method_order_fixed = list(model_roots.keys())
label_order_fixed = ["\n".join([part for part in str(m).split("-") if part != ""]) for m in method_order_fixed]


def method_label_from_underscores(name: str) -> str:
    text = str(name)
    if "-" in text:
        return "\n".join([part for part in text.split("-") if part != ""])
    return text


def prepare_plot_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["method_label"] = out["method"].map(method_label_from_underscores)
    return out


def format_axis_ticks(ax, tick_fontsize=10, rotation=45, linespacing=1.25):
    ax.tick_params(axis="x", labelsize=tick_fontsize, rotation=rotation)
    ax.tick_params(axis="y", labelsize=tick_fontsize)
    for tick in ax.get_xticklabels():
        tick.set_ha("center")
        tick.set_va("top")
        tick.set_linespacing(linespacing)


def annotate_counts_with_pct_on_boxes(
    ax,
    df_filtered: pd.DataFrame,
    method_order: list[str],
    df_total: pd.DataFrame,
):
    if df_filtered.empty or df_total.empty:
        return

    y_min, y_max = ax.get_ylim()
    y_span = max(y_max - y_min, 1e-8)
    offset = y_span * 0.03

    filtered_count_map = df_filtered.groupby("method").size().to_dict()
    total_count_map = df_total.groupby("method").size().to_dict()
    max_map = df_filtered.groupby("method")["scRMSD"].max().to_dict()

    for idx, method in enumerate(method_order):
        n_filtered = int(filtered_count_map.get(method, 0))
        n_total = int(total_count_map.get(method, 0))
        pct = (n_filtered / n_total * 100.0) if n_total > 0 else 0.0
        y_ref = max_map.get(method, y_min)
        y_text = y_ref + offset
        ax.text(
            idx,
            y_text,
            f"n={n_filtered} ({pct:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=max(tick_label_fontsize - 1, 8),
        )


def draw_global_box(
    df_for_plot: pd.DataFrame,
    title: str,
    out_path,
    annotate_filtered_stats: bool = False,
    df_total_for_pct: pd.DataFrame | None = None,
):
    if df_for_plot.empty:
        print(f"跳过绘图（无数据）: {title}")
        return

    plt.figure(figsize=(12, 7))
    ax = sns.boxplot(data=df_for_plot, x="method_label", y="scRMSD", order=label_order_fixed)
    sns.stripplot(
        data=df_for_plot,
        x="method_label",
        y="scRMSD",
        order=label_order_fixed,
        color="black",
        alpha=0.35,
        size=2.5,
        ax=ax,
    )
    ax.set_title(title, fontsize=plot_title_fontsize)
    ax.set_xlabel("Method", fontsize=axis_title_fontsize)
    ax.set_ylabel("scRMSD", fontsize=axis_title_fontsize)
    format_axis_ticks(ax, tick_fontsize=tick_label_fontsize, rotation=x_tick_rotation, linespacing=1.25)

    if annotate_filtered_stats:
        if df_total_for_pct is None:
            raise ValueError("annotate_filtered_stats=True 时，df_total_for_pct 不能为空")
        annotate_counts_with_pct_on_boxes(ax, df_for_plot, method_order_fixed, df_total_for_pct)

    plt.tight_layout()
    plt.savefig(out_path, dpi=250)
    plt.close()


if test_only_single_pdb:
    if not test_parent_pdb:
        raise ValueError("test_only_single_pdb=True 时，test_parent_pdb 不能为空")
    available = set(valid_df["parent_pdb"].unique())
    if test_parent_pdb not in available:
        raise ValueError(f"test_parent_pdb={test_parent_pdb} 不在 valid_df 中")
    parent_pdbs = [test_parent_pdb]
    print(f"[TEST MODE] 仅绘制父PDB: {test_parent_pdb}")
else:
    parent_pdbs = sorted(valid_df["parent_pdb"].unique())

plot_df = valid_df[valid_df["parent_pdb"].isin(parent_pdbs)].copy()
plot_df = prepare_plot_df(plot_df)

# 统一method显示顺序（按输入字典顺序）
method_order = method_order_fixed
label_order = label_order_fixed

# ========= 构建四种 ranking dataframe（所有模式都计算，供 Step4 使用） =========
# 数值类型转换
_plot_df = plot_df.copy()
_plot_df["scRMSD"] = pd.to_numeric(_plot_df["scRMSD"], errors="coerce")
_plot_df["ranking_score"] = pd.to_numeric(_plot_df["ranking_score"], errors="coerce")
_plot_df["pep_plddt"] = pd.to_numeric(_plot_df["pep_plddt"], errors="coerce")
_plot_df["iptm"] = pd.to_numeric(_plot_df["iptm"], errors="coerce")

# oracle: 最佳 scRMSD
oracle_df = (
    _plot_df.sort_values("scRMSD", ascending=True)
    .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
    .copy()
)
oracle_df = prepare_plot_df(oracle_df)

# rank1_ranking_score: 最佳 ranking_score
top1_df = (
    _plot_df.dropna(subset=["ranking_score"])
    .sort_values("ranking_score", ascending=False)
    .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
    .copy()
)
top1_df = prepare_plot_df(top1_df)

# rank1_pep_plddt: 最佳 pep_plddt
top1_pep_plddt_df = (
    _plot_df.dropna(subset=["pep_plddt"])
    .sort_values("pep_plddt", ascending=False)
    .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
    .copy()
)
top1_pep_plddt_df = prepare_plot_df(top1_pep_plddt_df)

# rank1_iptm: 最佳 iptm
top1_iptm_df = (
    _plot_df.dropna(subset=["iptm"])
    .sort_values("iptm", ascending=False)
    .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
    .copy()
)
top1_iptm_df = prepare_plot_df(top1_iptm_df)

# ========= 绘图（仅 step3_mode="full" 时执行） =========
if step3_mode == "df_only":
    print("[Step3 df_only mode] DataFrames built, skipping plots.")
    print(f"oracle_df: {len(oracle_df)} rows")
    print(f"top1_df (ranking_score): {len(top1_df)} rows")
    print(f"top1_pep_plddt_df: {len(top1_pep_plddt_df)} rows")
    print(f"top1_iptm_df: {len(top1_iptm_df)} rows")
else:
    # 图A：每个父PDB一张图（x=method_label, y=scRMSD）
    per_pdb_dir = figures_dir / f"per_pdb_{run_tag}"
    per_pdb_dir.mkdir(parents=True, exist_ok=True)

    for pdb in parent_pdbs:
        sub = plot_df[plot_df["parent_pdb"] == pdb].copy()
        if sub.empty:
            continue

        box_width = 0.5

        plt.figure(figsize=(12, 7))
        ax = sns.boxplot(
            data=sub,
            x="method_label",
            y="scRMSD",
            order=label_order,
            width=box_width,
        )
        sns.stripplot(
            data=sub,
            x="method_label",
            y="scRMSD",
            order=label_order,
            color="black",
            alpha=0.45,
            size=3,
            ax=ax,
        )
        ax.set_title(f"scRMSD distribution by method | {pdb}", fontsize=plot_title_fontsize)
        ax.set_xlabel("Method", fontsize=axis_title_fontsize)
        ax.set_ylabel("scRMSD", fontsize=axis_title_fontsize)
        format_axis_ticks(ax, tick_fontsize=tick_label_fontsize, rotation=x_tick_rotation, linespacing=1.25)
        plt.tight_layout()
        out_png = per_pdb_dir / f"{pdb}.scRMSD_boxplot.png"
        plt.savefig(out_png, dpi=200)
        plt.close()

    # ---------- 全局总图：6张 ----------
    top1_png = figures_dir / f"top1_ranking_score_scRMSD_distribution_{run_tag}.png"
    draw_global_box(top1_df, "Top1 Ranking Score scRMSD distribution", top1_png)

    oracle_png = figures_dir / f"oracle_scRMSD_distribution_{run_tag}.png"
    draw_global_box(oracle_df, "Oracle scRMSD distribution", oracle_png)

    top1_pep_plddt_png = figures_dir / f"top1_pep_plddt_scRMSD_distribution_{run_tag}.png"
    draw_global_box(top1_pep_plddt_df, "Top1 pep_plddt scRMSD distribution", top1_pep_plddt_png)

    top1_iptm_png = figures_dir / f"top1_iptm_scRMSD_distribution_{run_tag}.png"
    draw_global_box(top1_iptm_df, "Top1 iptm scRMSD distribution", top1_iptm_png)

    # <=2.5A 版本
    top1_df_le25 = top1_df[top1_df["scRMSD"] <= 2.5].copy()
    top1_le25_png = figures_dir / f"top1_ranking_score_scRMSD_le2.5A_distribution_{run_tag}.png"
    draw_global_box(
        top1_df_le25,
        "Top1 Ranking Score scRMSD distribution (<=2.5A)",
        top1_le25_png,
        annotate_filtered_stats=True,
        df_total_for_pct=top1_df,
    )

    oracle_df_le25 = oracle_df[oracle_df["scRMSD"] <= 2.5].copy()
    oracle_le25_png = figures_dir / f"oracle_scRMSD_le2.5A_distribution_{run_tag}.png"
    draw_global_box(
        oracle_df_le25,
        "Oracle scRMSD distribution (<=2.5A)",
        oracle_le25_png,
        annotate_filtered_stats=True,
        df_total_for_pct=oracle_df,
    )

    top1_pep_plddt_df_le25 = top1_pep_plddt_df[top1_pep_plddt_df["scRMSD"] <= 2.5].copy()
    top1_pep_plddt_le25_png = figures_dir / f"top1_pep_plddt_scRMSD_le2.5A_distribution_{run_tag}.png"
    draw_global_box(
        top1_pep_plddt_df_le25,
        "Top1 pep_plddt scRMSD distribution (<=2.5A)",
        top1_pep_plddt_le25_png,
        annotate_filtered_stats=True,
        df_total_for_pct=top1_pep_plddt_df,
    )

    top1_iptm_df_le25 = top1_iptm_df[top1_iptm_df["scRMSD"] <= 2.5].copy()
    top1_iptm_le25_png = figures_dir / f"top1_iptm_scRMSD_le2.5A_distribution_{run_tag}.png"
    draw_global_box(
        top1_iptm_df_le25,
        "Top1 iptm scRMSD distribution (<=2.5A)",
        top1_iptm_le25_png,
        annotate_filtered_stats=True,
        df_total_for_pct=top1_iptm_df,
    )

    print(f"Per-PDB figures dir: {per_pdb_dir}")
    print(f"Top1 Ranking Score Global figure: {top1_png}")
    print(f"Oracle Global figure: {oracle_png}")
    print(f"Top1 pep_plddt Global figure: {top1_pep_plddt_png}")
    print(f"Top1 iptm Global figure: {top1_iptm_png}")
    print(f"Top1 <=2.5A Global figure: {top1_le25_png}")
    print(f"Oracle <=2.5A Global figure: {oracle_le25_png}")
    print(f"Top1 pep_plddt <=2.5A Global figure: {top1_pep_plddt_le25_png}")
    print(f"Top1 iptm <=2.5A Global figure: {top1_iptm_le25_png}")

# 附加：输出方法级汇总统计（保留在exports按日期目录）
method_stats = (
    plot_df.groupby("method")["scRMSD"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)
method_stats_path = exports_date_dir / f"method_scRMSD_stats_{run_tag}.csv"
method_stats.to_csv(method_stats_path, index=False)

print(f"Method stats: {method_stats_path}")
print(method_stats)


[Step3 df_only mode] DataFrames built, skipping plots.
oracle_df: 1530 rows
top1_df (ranking_score): 1530 rows
top1_pep_plddt_df: 1530 rows
top1_iptm_df: 1530 rows
Method stats: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/exports/20260327/method_scRMSD_stats_20260327_152811.csv
                                   method  count       mean    median  \
0    AlphaFold3_v3.0.1-pro_msa_notemplate   2550   4.734246   1.92520   
1      AlphaFold3_v3.0.1-pro_msa_template   2550   4.915429   2.00745   
2  AlphaFold3_v3.0.1-pro_nomsa_notemplate   2550  21.289131  20.18540   
3    AlphaFold3_v3.0.1-pro_nomsa_template   2550   7.764452   2.82940   
4      Protenix_v0.5.0-pro_msa_notemplate   2550   6.282725   2.26135   
5      Protenix_v1.0.0-pro_msa_notemplate   2550   4.737282   1.74850   
6        Protenix_v1.0.0-pro_msa_template   2550   4.735416   1.70580   
7    Protenix_v1.0.0-pro_nomsa_notemplate   2550  21.770708  21.79160   
8      Protenix_v1.0.0-pro_nomsa_template   2550 

In [6]:
# ========= Step 4: 按template格式导出总图CSV（首列方法，先oracle后rank1） =========
global_csv_dir = csv_summary_dir / f"global_plot_data_{run_tag}"
global_csv_dir.mkdir(parents=True, exist_ok=True)

# 数据来源开关："from_df" 或 "from_csv_summary_dict"
export_source_mode = "from_df"

# 当 export_source_mode="from_csv_summary_dict" 时启用
# 说明：键必须是方法名（建议与model_roots键一致），值是csv路径（可用相对csv_summary_dir）
custom_source_dict = {
    # "Protenix_v1.0.0-pro_nomsa_notemplate-pep_nomsa_notemplate": "Protenix_v1.0.0-pro_nomsa_notemplate-pep_nomsa_notemplate/xxxx.metrics_summary.csv"
}


def _ensure_complex_seed_id(df: pd.DataFrame) -> pd.Series:
    if "complex_seed_id" in df.columns:
        return df["complex_seed_id"].astype(str)
    needed = ["complex", "seed", "id"]
    if all(col in df.columns for col in needed):
        return (
            df["complex"].astype(str)
            + "_"
            + df["seed"].astype(str)
            + "_"
            + df["id"].astype(str)
        )
    raise ValueError("无法构建 complex_seed_id：缺少 complex/seed/id 或 complex_seed_id")


def _build_selected_from_raw(raw_df: pd.DataFrame):
    df = raw_df.copy()
    need_cols = {"method", "parent_pdb", "scRMSD"}
    miss = sorted(need_cols - set(df.columns))
    if miss:
        raise ValueError(f"原始数据缺少必要列: {miss}")

    if "ranking_score" not in df.columns:
        df["ranking_score"] = pd.NA

    df["scRMSD"] = pd.to_numeric(df["scRMSD"], errors="coerce")
    df["ranking_score"] = pd.to_numeric(df["ranking_score"], errors="coerce")
    df = df.dropna(subset=["scRMSD"]).copy()
    df["complex_seed_id"] = _ensure_complex_seed_id(df)

    oracle_df_local = (
        df.sort_values("scRMSD", ascending=True)
        .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
        .copy()
    )

    top1_src = df.dropna(subset=["ranking_score"]).copy()
    top1_df_local = (
        top1_src.sort_values("ranking_score", ascending=False)
        .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
        .copy()
    )

    # rank1_pep_plddt: best pep_plddt per (method, parent_pdb)
    pep_plddt_src = df.dropna(subset=["pep_plddt"]).copy()
    rank1_pep_plddt_df_local = (
        pep_plddt_src.sort_values("pep_plddt", ascending=False)
        .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
        .copy()
    )

    # rank1_iptm: best iptm per (method, parent_pdb)
    iptm_src = df.dropna(subset=["iptm"]).copy()
    rank1_iptm_df_local = (
        iptm_src.sort_values("iptm", ascending=False)
        .drop_duplicates(subset=["method", "parent_pdb"], keep="first")
        .copy()
    )

    return oracle_df_local, top1_df_local, rank1_pep_plddt_df_local, rank1_iptm_df_local


def _load_raw_from_custom_dict(mapping: dict) -> pd.DataFrame:
    if not mapping:
        raise ValueError("custom_source_dict 为空，无法从csv_summary_dict模式读取数据")

    frames = []
    for method_name, csv_path in mapping.items():
        p = Path(csv_path)
        if not p.is_absolute():
            p = csv_summary_dir / p
        if not p.exists():
            print(f"[WARN] 文件不存在，跳过: {p}")
            continue

        sub = pd.read_csv(p)
        sub = sub.copy()
        sub["method"] = method_name

        if "parent_pdb" not in sub.columns:
            parent_guess = p.name.replace(".metrics_summary.csv", "")
            sub["parent_pdb"] = parent_guess

        frames.append(sub)

    if not frames:
        raise RuntimeError("custom_source_dict 未读取到任何有效CSV")

    return pd.concat(frames, ignore_index=True)


def export_template_all_rankings(
    oracle_df_in: pd.DataFrame,
    top1_df_in: pd.DataFrame,
    rank1_pep_plddt_df_in: pd.DataFrame,
    rank1_iptm_df_in: pd.DataFrame,
    out_csv: Path,
):
    method_order = list(model_roots.keys())

    oracle_df_local = oracle_df_in.copy()
    top1_df_local = top1_df_in.copy()
    rank1_pep_plddt_df_local = rank1_pep_plddt_df_in.copy()
    rank1_iptm_df_local = rank1_iptm_df_in.copy()

    oracle_df_local["scRMSD"] = pd.to_numeric(oracle_df_local["scRMSD"], errors="coerce")
    top1_df_local["scRMSD"] = pd.to_numeric(top1_df_local["scRMSD"], errors="coerce")
    rank1_pep_plddt_df_local["scRMSD"] = pd.to_numeric(rank1_pep_plddt_df_local["scRMSD"], errors="coerce")
    rank1_iptm_df_local["scRMSD"] = pd.to_numeric(rank1_iptm_df_local["scRMSD"], errors="coerce")

    oracle_series_map = {}
    rank1_series_map = {}
    rank1_pep_plddt_series_map = {}
    rank1_iptm_series_map = {}

    max_oracle_len = 0
    max_rank1_len = 0
    max_rank1_pep_plddt_len = 0
    max_rank1_iptm_len = 0

    for method in method_order:
        oracle_vals = (
            oracle_df_local.loc[oracle_df_local["method"] == method]
            .sort_values(["parent_pdb", "scRMSD"], na_position="last")
            ["scRMSD"]
            .dropna()
            .tolist()
        )
        rank1_vals = (
            top1_df_local.loc[top1_df_local["method"] == method]
            .sort_values(["parent_pdb", "scRMSD"], na_position="last")
            ["scRMSD"]
            .dropna()
            .tolist()
        )
        rank1_pep_plddt_vals = (
            rank1_pep_plddt_df_local.loc[rank1_pep_plddt_df_local["method"] == method]
            .sort_values(["parent_pdb", "scRMSD"], na_position="last")
            ["scRMSD"]
            .dropna()
            .tolist()
        )
        rank1_iptm_vals = (
            rank1_iptm_df_local.loc[rank1_iptm_df_local["method"] == method]
            .sort_values(["parent_pdb", "scRMSD"], na_position="last")
            ["scRMSD"]
            .dropna()
            .tolist()
        )

        oracle_series_map[method] = oracle_vals
        rank1_series_map[method] = rank1_vals
        rank1_pep_plddt_series_map[method] = rank1_pep_plddt_vals
        rank1_iptm_series_map[method] = rank1_iptm_vals

        max_oracle_len = max(max_oracle_len, len(oracle_vals))
        max_rank1_len = max(max_rank1_len, len(rank1_vals))
        max_rank1_pep_plddt_len = max(max_rank1_pep_plddt_len, len(rank1_pep_plddt_vals))
        max_rank1_iptm_len = max(max_rank1_iptm_len, len(rank1_iptm_vals))

    max_len = max(max_oracle_len, max_rank1_len, max_rank1_pep_plddt_len, max_rank1_iptm_len)

    out_rows = []
    for method in method_order:
        oracle_vals = oracle_series_map.get(method, [])
        rank1_vals = rank1_series_map.get(method, [])
        rank1_pep_plddt_vals = rank1_pep_plddt_series_map.get(method, [])
        rank1_iptm_vals = rank1_iptm_series_map.get(method, [])

        oracle_pad = oracle_vals + [pd.NA] * (max_len - len(oracle_vals))
        rank1_pad = rank1_vals + [pd.NA] * (max_len - len(rank1_vals))
        rank1_pep_plddt_pad = rank1_pep_plddt_vals + [pd.NA] * (max_len - len(rank1_pep_plddt_vals))
        rank1_iptm_pad = rank1_iptm_vals + [pd.NA] * (max_len - len(rank1_iptm_vals))

        out_rows.append([method] + oracle_pad + rank1_pad + rank1_pep_plddt_pad + rank1_iptm_pad)

    out_cols = (
        [""]
        + (["oracle"] * max_len)
        + (["rank1_ranking_score"] * max_len)
        + (["rank1_pep_plddt"] * max_len)
        + (["rank1_iptm"] * max_len)
    )
    out_df = pd.DataFrame(out_rows, columns=out_cols)
    out_df.to_csv(out_csv, index=False)

    print(f"Series max lengths — oracle: {max_oracle_len}, rank1_ranking_score: {max_rank1_len}, "
          f"rank1_pep_plddt: {max_rank1_pep_plddt_len}, rank1_iptm: {max_rank1_iptm_len}")
    return out_df


if export_source_mode == "from_df":
    required_global_vars = ["oracle_df", "top1_df", "top1_pep_plddt_df", "top1_iptm_df"]
    missing_vars = [v for v in required_global_vars if v not in globals()]
    if missing_vars:
        raise RuntimeError(f"from_df 模式缺少变量: {missing_vars}。请先运行 Step3 和 Step4 单元。")

    # 直接使用 Step3 中已计算好的 dataframe（均为 prepare_plot_df 后的版本）
    oracle_src = oracle_df.copy()
    top1_src = top1_df.copy()
    rank1_pep_plddt_src = top1_pep_plddt_df.copy()
    rank1_iptm_src = top1_iptm_df.copy()

    # 补充 complex_seed_id（Step3 未计算此列）
    oracle_src["complex_seed_id"] = _ensure_complex_seed_id(oracle_src)
    top1_src["complex_seed_id"] = _ensure_complex_seed_id(top1_src)
    rank1_pep_plddt_src["complex_seed_id"] = _ensure_complex_seed_id(rank1_pep_plddt_src)
    rank1_iptm_src["complex_seed_id"] = _ensure_complex_seed_id(rank1_iptm_src)

elif export_source_mode == "from_csv_summary_dict":
    raw_custom_df = _load_raw_from_custom_dict(custom_source_dict)
    oracle_src, top1_src, rank1_pep_plddt_src, rank1_iptm_src = _build_selected_from_raw(raw_custom_df)

else:
    raise ValueError("export_source_mode 仅支持 'from_df' 或 'from_csv_summary_dict'")


template_csv = global_csv_dir / f"template_style_oracle_rank1_{run_tag}.csv"
template_df = export_template_all_rankings(oracle_src, top1_src, rank1_pep_plddt_src, rank1_iptm_src, template_csv)

# 新增：输出 RMSD<=2.5 的过滤版 global template csv
oracle_src_le25 = oracle_src[pd.to_numeric(oracle_src["scRMSD"], errors="coerce") <= 2.5].copy()
top1_src_le25 = top1_src[pd.to_numeric(top1_src["scRMSD"], errors="coerce") <= 2.5].copy()
rank1_pep_plddt_src_le25 = rank1_pep_plddt_src[pd.to_numeric(rank1_pep_plddt_src["scRMSD"], errors="coerce") <= 2.5].copy()
rank1_iptm_src_le25 = rank1_iptm_src[pd.to_numeric(rank1_iptm_src["scRMSD"], errors="coerce") <= 2.5].copy()
template_le25_csv = global_csv_dir / f"template_style_oracle_rank1_le2.5A_{run_tag}.csv"
template_le25_df = export_template_all_rankings(
    oracle_src_le25, top1_src_le25, rank1_pep_plddt_src_le25, rank1_iptm_src_le25, template_le25_csv
)

print(f"Source mode: {export_source_mode}")
print(f"Saved template csv: {template_csv}")
print(f"Template shape: {template_df.shape}")
print(f"Saved <=2.5A template csv: {template_le25_csv}")
print(f"<=2.5A Template shape: {template_le25_df.shape}")
print(f"Global plot data dir: {global_csv_dir}")


Series max lengths — oracle: 170, rank1_ranking_score: 170, rank1_pep_plddt: 170, rank1_iptm: 170
Series max lengths — oracle: 131, rank1_ranking_score: 110, rank1_pep_plddt: 111, rank1_iptm: 110
Source mode: from_df
Saved template csv: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/csv_summary/20260327/global_plot_data_20260327_152811/template_style_oracle_rank1_20260327_152811.csv
Template shape: (9, 681)
Saved <=2.5A template csv: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/csv_summary/20260327/global_plot_data_20260327_152811/template_style_oracle_rank1_le2.5A_20260327_152811.csv
<=2.5A Template shape: (9, 525)
Global plot data dir: /QIN/junjiechen/250401-Dpepalign/Benchmark/predict_summary/csv_summary/20260327/global_plot_data_20260327_152811


In [ ]:
# 根据csv中的内容，编写代码，读取第二行开始每一行相同列名的非空个数

import pandas as pd                                                                                                                                                 
 
csv_path = "csv_summary/20260327/global_plot_data_20260327_152811/template_style_oracle_rank1_le2.5A_20260327_152811.csv"                                           
df = pd.read_csv(csv_path)
                                                                                                                                                                    
# 按原始列名分组
groups = {}
for col in df.columns[1:]:                                                                                                                                          
    name = col.rsplit(".", 1)[0]
    groups.setdefault(name, []).append(col)                                                                                                                         
                
# 统计每行                                                                                                                                                          
for idx, row in df.iterrows():
    method = row[df.columns[0]]                                                                                                                                     
    counts = {name: row[cols].notna().sum() for name, cols in groups.items()}
    print(f"{method}: {counts}")       

总行数(含表头): 10, 数据行数: 9
列名分组: {'oracle': 131, 'rank1_ranking_score': 131, 'rank1_pep_plddt': 131, 'rank1_iptm': 131}

AlphaFold3_v3.0.1-pro_msa_template: {'oracle': 0.7705882352941177, 'rank1_ranking_score': 0.5941176470588235, 'rank1_pep_plddt': 0.6, 'rank1_iptm': 0.5705882352941176}
AlphaFold3_v3.0.1-pro_msa_notemplate: {'oracle': 0.7529411764705882, 'rank1_ranking_score': 0.611764705882353, 'rank1_pep_plddt': 0.5882352941176471, 'rank1_iptm': 0.6}
AlphaFold3_v3.0.1-pro_nomsa_template: {'oracle': 0.6058823529411764, 'rank1_ranking_score': 0.4588235294117647, 'rank1_pep_plddt': 0.4823529411764706, 'rank1_iptm': 0.47058823529411764}
AlphaFold3_v3.0.1-pro_nomsa_notemplate: {'oracle': 0.15294117647058825, 'rank1_ranking_score': 0.12941176470588237, 'rank1_pep_plddt': 0.12352941176470589, 'rank1_iptm': 0.11764705882352941}
Protenix_v1.0.0-pro_msa_template: {'oracle': 0.7235294117647059, 'rank1_ranking_score': 0.6470588235294118, 'rank1_pep_plddt': 0.6529411764705882, 'rank1_iptm': 0.6470588